# Prodigy_Master_RERUN_v2 — Orchestrator finale

Esegue in sequenza i 6 sub-notebook v2 su Azure. Ogni sub-notebook è idempotente: skippa run già completati. In caso di kernel timeout, basta rilanciare Cell 2.

## Ordine di esecuzione (R27_v2 audit a fine, dopo TUTTI i training)

1. `Prodigy_Ablation_Study_R25_v2.ipynb` (18 run, ~2h)
2. `Prodigy_Fusion_Study_R26_v2.ipynb` (6 run, ~40 min)
3. `Prodigy_Tuning_R28_v2.ipynb` (5 run, ~30 min)
4. `Prodigy_DecoderFix_R29_v2.ipynb` (12 run, ~2h)
5. `Prodigy_Identifiability_R30.ipynb` (9 run, ~1h)
6. `Prodigy_Audit_R27_v2.ipynb` (audit di TUTTI i 50 run sopra, ~15 min)

Totale: 50 run + 1 audit, ~6-8h Azure CPU.

**Nota ordine**: R27_v2 audit è L'ULTIMO perché serve checkpoint di TUTTI i training (R25/R26/R28/R29/R30). Solo a fine può misurare T_intra_corr + rank_effective sui best_model.pt di ogni run.

## Meta-analisi finale (Cell 3)

Consolida i 5 aggregator CSV + audit summary e fornisce verdetto su:
- Quale run ha T_intra > 0.1 (sblocco rank-collapse)?
- Quale combinazione (ablation × decoder × identifiability) è il nuovo champion?

In [ ]:
# Cell 1 -- ENV check + sub-notebook list (ORDINE: training prima, audit ULTIMO)
import sys, os, subprocess
BRANCH = 'Prodigy_Deep_Study'
SUB_NOTEBOOKS = [
    'Prodigy_Ablation_Study_R25_v2.ipynb',
    'Prodigy_Fusion_Study_R26_v2.ipynb',
    'Prodigy_Tuning_R28_v2.ipynb',
    'Prodigy_DecoderFix_R29_v2.ipynb',
    'Prodigy_Identifiability_R30.ipynb',
    'Prodigy_Audit_R27_v2.ipynb',           # ← ULTIMO: audit dopo tutti i training
]
for nb in SUB_NOTEBOOKS:
    assert os.path.isfile(nb), f'MISSING: {nb}'
print(f'  [OK] {len(SUB_NOTEBOOKS)} sub-notebook presenti (audit ultimo)')
br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
assert br == BRANCH
print(f'  [OK] branch={br}')

In [ ]:
# Cell 2 -- Esegue i 6 sub-notebook in sequenza (via nbconvert)
import subprocess, sys, time, os

t_total = time.time()
for nb in SUB_NOTEBOOKS:
    print(f'\n{"="*78}\nEXECUTING: {nb}\n{"="*78}')
    t0 = time.time()
    cmd = [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook',
           '--execute', '--inplace', nb,
           '--ExecutePreprocessor.timeout=-1',
           '--ExecutePreprocessor.kernel_name=python3']
    r = subprocess.run(cmd, capture_output=False)
    el = time.time() - t0
    if r.returncode != 0:
        print(f'\n[FAIL] {nb} exit={r.returncode}. Verifica output e rilancia.')
        break
    print(f'\n[OK] {nb} done in {el/60:.1f} min')

print(f'\n{"="*78}\nMASTER RERUN done in {(time.time()-t_total)/60:.0f} min')

In [ ]:
# Cell 3 -- META-ANALISI cross-studies + verdetto finale
import os, pandas as pd, numpy as np
from IPython.display import display, Markdown

STUDIES = [
    ('R25 Ablation v2',    'results/Prodigy_Study/R25_Ablation_v2/_aggregate.csv'),
    ('R26 Fusion v2',      'results/Prodigy_Study/R26_Fusion_v2/_aggregate.csv'),
    ('R28 ProdigyTuning v2','results/Prodigy_Study/R28_ProdigyTuning_v2/_aggregate.csv'),
    ('R29 DecoderFix v2',  'results/Prodigy_Study/R29_DecoderFix_v2/_aggregate.csv'),
    ('R30 Identifiability','results/Prodigy_Study/R30_Identifiability/_aggregate.csv'),
]

all_dfs = []
for name, p in STUDIES:
    if not os.path.isfile(p):
        print(f'  [MISSING] {p}')
        continue
    df = pd.read_csv(p)
    df['study'] = name
    all_dfs.append(df)
big = pd.concat(all_dfs, ignore_index=True)
print(f'Total run analizzati: {len(big)}')
print(f'Clean runs (gn_max < 25): {(big.gn_max_preclip < 25).sum()}/{len(big)}')

display(Markdown('## Top 10 cross-studies per T_intra_corr'))
top10 = big.sort_values('T_intra_corr', ascending=False).head(10)
display(top10[['study','tag','val_data','T_tracking_corr','T_intra_corr','gn_max_preclip']].round(4))

display(Markdown('## Verdetto sblocco T-tracking'))
n_intra_strong = (big.T_intra_corr > 0.10).sum()
n_intra_med    = (big.T_intra_corr > 0.058).sum()
print(f'Run con T_intra > 0.10: {n_intra_strong}')
print(f'Run con T_intra > 0.058 (top R27 historic): {n_intra_med}')
if n_intra_strong > 0:
    best = big.sort_values('T_intra_corr', ascending=False).iloc[0]
    display(Markdown(f'### ✅ SBLOCCO T-tracking: {best.study} {best.tag} = T_intra {best.T_intra_corr:.3f}. NUOVO CHAMPION.'))
elif n_intra_med > 0:
    display(Markdown(f'### ⚠ Miglioramento moderato. {n_intra_med} run > 0.058 (top historic).'))
else:
    display(Markdown('### ❌ Nessun sblocco. Identifiability rimane problema strutturale -- forse 864p insufficiente.'))

# Save consolidated
big.to_csv('results/Prodigy_Study/_FINAL_v2_consolidated.csv', index=False)
print(f'\nSalvato: results/Prodigy_Study/_FINAL_v2_consolidated.csv')